# Geometric EEG SSL — Colab Pretraining Notebook

**What this does:**
1. Installs dependencies
2. Mounts Google Drive (checkpoints saved there — survive session disconnects)
3. Arms a keep-alive against laptop sleep + idle disconnect
4. Clones the repo
5. Downloads all three datasets: PhysioNet MI, BCIC-2B, Sleep-EDFx
6. Pretrains all 5 variants: G1, G2, G3, Transductive Codex, Channel-Independent
7. Runs eval experiments E1, E2(a/b/c), E5, E7 via the orchestration scripts

**Before running:** Runtime → Change runtime type → GPU (T4 free, A100 Colab Pro)

---

## 0. GPU check

In [ ]:
import subprocess, sys
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
if result.returncode == 0:
    print('GPU:', result.stdout.strip())
else:
    print('WARNING: no GPU detected — set Runtime → Change runtime type → GPU')
    sys.exit(1)

## 1. Install dependencies

In [ ]:
%%capture
!pip install mne moabb scikit-learn pyyaml scipy

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/geometric_eeg_ssl'
os.makedirs(DRIVE_ROOT, exist_ok=True)

# MNE data cached here — avoids re-downloading across sessions
MNE_DATA_DIR = f'{DRIVE_ROOT}/mne_data'
os.makedirs(MNE_DATA_DIR, exist_ok=True)
os.environ['MNE_DATA'] = MNE_DATA_DIR

# Checkpoints saved here
CKPT_ROOT = f'{DRIVE_ROOT}/runs'
os.makedirs(CKPT_ROOT, exist_ok=True)
print(f'Drive mounted. Checkpoints → {CKPT_ROOT}')

## 2b. Keep Colab alive while your laptop sleeps

Colab sessions are tied to the browser tab that owns them. If the laptop sleeps
or the network drops, the WebSocket dies and the runtime can be reclaimed even
though GPU compute is still busy. The training loop saves a checkpoint every 10
epochs, so a disconnect costs at most ~10 epochs — but it's better to avoid one
entirely. Three layers of protection, applied together:

**1. Stop the laptop from sleeping.**
- **macOS:** open Terminal and run `caffeinate -dis &` before closing the lid.
  Or System Settings → Battery → "Prevent automatic sleeping on power adapter
  when the display is off" + plug in.
- **Windows:** Settings → System → Power → Screen and sleep → set "When plugged
  in, put my device to sleep" to *Never*.

**2. Suppress Colab's idle prompt (cell below).**
Colab pops up a "Are you still here?" dialog after ~90 min of UI inactivity.
The JS snippet below auto-clicks the connect button every minute.

**3. Trust the resume cell.**
Each pretrain section has its own RESUME cell that finds the latest checkpoint
and restarts from there.

In [ ]:
from IPython.display import display, Javascript
display(Javascript('''
function ClickConnect() {
  const btn = document.querySelector("colab-connect-button");
  if (btn && btn.shadowRoot) {
    const inner = btn.shadowRoot.querySelector("#connect");
    if (inner) inner.click();
  }
  console.log("colab keep-alive ping " + new Date().toLocaleTimeString());
}
if (window._colabKeepAlive) clearInterval(window._colabKeepAlive);
window._colabKeepAlive = setInterval(ClickConnect, 60000);
console.log("colab keep-alive armed (60s interval)");
'''))
print('Keep-alive armed. Re-run this cell after any browser refresh.')

## 3. Clone repo

In [ ]:
import os, sys
REPO_DIR = '/content/geometric-eeg-ssl'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/tianxin-scu/geometric-eeg-ssl.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull
sys.path.insert(0, REPO_DIR)
sys.path.insert(0, f'{REPO_DIR}/src')
print('Repo ready at', REPO_DIR)

## 4a. Download PhysioNet MI data

First run takes ~20 min. Subsequent runs load from Drive instantly.
Resume-aware: fully-cached subjects are skipped.

In [ ]:
import os, mne
mne.set_log_level('WARNING')

EXCLUDED = {88, 92, 100, 104}
ALL_SUBJECTS_MI = [s for s in range(1, 110) if s not in EXCLUDED]  # 105 subjects
MI_RUNS = [4, 6, 8, 10, 12, 14]

EEGBCI_ROOT = os.path.join(MNE_DATA_DIR, 'MNE-eegbci-data', 'files', 'eegmmidb', '1.0.0')

def subject_fully_cached(subj, runs):
    subj_dir = os.path.join(EEGBCI_ROOT, f'S{subj:03d}')
    if not os.path.isdir(subj_dir):
        return False
    return all(
        os.path.exists(os.path.join(subj_dir, f'S{subj:03d}R{run:02d}.edf')) and
        os.path.getsize(os.path.join(subj_dir, f'S{subj:03d}R{run:02d}.edf')) > 0
        for run in runs
    )

cached = [s for s in ALL_SUBJECTS_MI if subject_fully_cached(s, MI_RUNS)]
todo = [s for s in ALL_SUBJECTS_MI if s not in cached]
print(f'PhysioNet MI: cached {len(cached)}/{len(ALL_SUBJECTS_MI)} subjects. Downloading {len(todo)}.')

for i, subj in enumerate(todo):
    try:
        mne.datasets.eegbci.load_data(subj, MI_RUNS, path=MNE_DATA_DIR, verbose=False)
    except Exception as e:
        print(f'  subject {subj:3d}: download failed ({e})')
        continue
    if (i + 1) % 10 == 0 or (i + 1) == len(todo):
        print(f'  downloaded {i+1}/{len(todo)} (subject {subj:3d})')

still_missing = [s for s in ALL_SUBJECTS_MI if not subject_fully_cached(s, MI_RUNS)]
if still_missing:
    print(f'WARNING: {len(still_missing)} subjects still incomplete: {still_missing}')
else:
    print(f'All {len(ALL_SUBJECTS_MI)} PhysioNet MI subjects ready.')

## 4b. Download BCIC-2B data

3-channel motor imagery dataset (C3, Cz, C4), 9 subjects. Used for
cross-montage zero-shot transfer in E2(c), E5, E7.
Download via MOABB (~30 MB, fast).

In [ ]:
import os
os.environ['MNE_DATA'] = MNE_DATA_DIR

from moabb.datasets import BNCI2014_004
import moabb
moabb.set_log_level('WARNING')

ds = BNCI2014_004()
print('Downloading BCIC-2B (BNCI2014_004)...')
try:
    ds.download(subject_list=list(range(1, 10)))
    print('BCIC-2B download complete.')
except Exception as e:
    # MOABB sometimes raises on partial cache; data may still be usable
    print(f'MOABB download reported: {e}')
    print('Attempting to load subject 1 to verify cache...')
    try:
        _ = ds.get_data(subjects=[1])
        print('Subject 1 loaded OK — cache is usable.')
    except Exception as e2:
        print(f'WARNING: could not load subject 1: {e2}')

## 4c. Download Sleep-EDFx data

2-channel sleep staging dataset (~78 subjects, 2 nights each). Used for
cross-session within-subject evaluation in E2(a).
First run downloads ~500 MB from PhysioNet; subsequent runs load from Drive.

In [ ]:
import os, mne
mne.set_log_level('WARNING')
os.environ['MNE_DATA'] = MNE_DATA_DIR

# MNE stores Sleep-EDFx under {MNE_DATA}/physionet-sleep-data/
_UNAVAILABLE = {39, 68, 69, 78, 79}
ALL_SUBJECTS_SLEEP = [s for s in range(0, 83) if s not in _UNAVAILABLE]
print(f'Sleep-EDFx: {len(ALL_SUBJECTS_SLEEP)} subjects to cache.')

n_done = 0
n_failed = 0
for subj in ALL_SUBJECTS_SLEEP:
    try:
        mne.datasets.sleep_physionet.age.fetch_data(
            subjects=[subj], recording=[1, 2], path=MNE_DATA_DIR, verbose=False
        )
        n_done += 1
    except Exception as e:
        n_failed += 1
        # Some subjects have only 1 recording — try night 1 only
        try:
            mne.datasets.sleep_physionet.age.fetch_data(
                subjects=[subj], recording=[1], path=MNE_DATA_DIR, verbose=False
            )
            n_done += 1
            n_failed -= 1
        except Exception:
            pass

print(f'Sleep-EDFx: {n_done} subjects cached, {n_failed} failed/unavailable.')

## 5. Pretrain — G1 (geometric, score-only bias)

~7–10 hrs on T4, ~2–3 hrs on A100. Checkpoints saved every 10 epochs to Drive.

In [ ]:
import os, glob

G1_CKPT_DIR = f'{CKPT_ROOT}/g1_full'
os.makedirs(G1_CKPT_DIR, exist_ok=True)
os.environ['MNE_DATA'] = MNE_DATA_DIR

# Auto-resumes from the latest checkpoint if one exists; starts fresh otherwise.
_mode = 'resuming' if glob.glob(f'{G1_CKPT_DIR}/epoch_*.pt') else 'starting fresh'
print(f'G1 (geometric, score-bias): {_mode}')

!python {REPO_DIR}/scripts/pretrain.py \\
    --config {REPO_DIR}/configs/pretrain/geometric_g1.yaml \\
    --ckpt-dir {G1_CKPT_DIR} \\
    --device cuda \\
    --resume latest \\
    2>&1 | tee -a {G1_CKPT_DIR}/train_log.txt

## 6. Pretrain — G2 (geometric, value-modulation)

Same duration estimate as G1. Run after G1 completes, or in a separate session.

In [ ]:
import os, glob

G2_CKPT_DIR = f'{CKPT_ROOT}/g2_full'
os.makedirs(G2_CKPT_DIR, exist_ok=True)
os.environ['MNE_DATA'] = MNE_DATA_DIR

# Auto-resumes from the latest checkpoint if one exists; starts fresh otherwise.
_mode = 'resuming' if glob.glob(f'{G2_CKPT_DIR}/epoch_*.pt') else 'starting fresh'
print(f'G2 (geometric, value-modulation): {_mode}')

!python {REPO_DIR}/scripts/pretrain.py \\
    --config {REPO_DIR}/configs/pretrain/geometric_g2.yaml \\
    --ckpt-dir {G2_CKPT_DIR} \\
    --device cuda \\
    --resume latest \\
    2>&1 | tee -a {G2_CKPT_DIR}/train_log.txt

## 7. Pretrain — G3 (geometric, score + value)

In [ ]:
import os, glob

G3_CKPT_DIR = f'{CKPT_ROOT}/g3_full'
os.makedirs(G3_CKPT_DIR, exist_ok=True)
os.environ['MNE_DATA'] = MNE_DATA_DIR

# Auto-resumes from the latest checkpoint if one exists; starts fresh otherwise.
_mode = 'resuming' if glob.glob(f'{G3_CKPT_DIR}/epoch_*.pt') else 'starting fresh'
print(f'G3 (geometric, score + value): {_mode}')

!python {REPO_DIR}/scripts/pretrain.py \\
    --config {REPO_DIR}/configs/pretrain/geometric_g3.yaml \\
    --ckpt-dir {G3_CKPT_DIR} \\
    --device cuda \\
    --resume latest \\
    2>&1 | tee -a {G3_CKPT_DIR}/train_log.txt

## 8. Pretrain — Transductive Codex

In [ ]:
import os, glob

CODEX_CKPT_DIR = f'{CKPT_ROOT}/codex_full'
os.makedirs(CODEX_CKPT_DIR, exist_ok=True)
os.environ['MNE_DATA'] = MNE_DATA_DIR

# Auto-resumes from the latest checkpoint if one exists; starts fresh otherwise.
_mode = 'resuming' if glob.glob(f'{CODEX_CKPT_DIR}/epoch_*.pt') else 'starting fresh'
print(f'Transductive Codex: {_mode}')

!python {REPO_DIR}/scripts/pretrain.py \\
    --config {REPO_DIR}/configs/pretrain/transductive_codex.yaml \\
    --ckpt-dir {CODEX_CKPT_DIR} \\
    --device cuda \\
    --resume latest \\
    2>&1 | tee -a {CODEX_CKPT_DIR}/train_log.txt

## 9. Pretrain — Channel-Independent baseline

No geometric attention, no codex. Matches G1 budget. Required for E7.

In [ ]:
import os, glob

CHIND_CKPT_DIR = f'{CKPT_ROOT}/chind_full'
os.makedirs(CHIND_CKPT_DIR, exist_ok=True)
os.environ['MNE_DATA'] = MNE_DATA_DIR

# Auto-resumes from the latest checkpoint if one exists; starts fresh otherwise.
_mode = 'resuming' if glob.glob(f'{CHIND_CKPT_DIR}/epoch_*.pt') else 'starting fresh'
print(f'Channel-Independent: {_mode}')

!python {REPO_DIR}/scripts/pretrain.py \\
    --config {REPO_DIR}/configs/pretrain/channel_independent.yaml \\
    --ckpt-dir {CHIND_CKPT_DIR} \\
    --device cuda \\
    --resume latest \\
    2>&1 | tee -a {CHIND_CKPT_DIR}/train_log.txt

## 9b. Verify checkpoint inventory

Run after all pretrains finish to confirm what's available before eval.

In [ ]:
import glob, os

VARIANTS = [
    ('G1',                 f'{CKPT_ROOT}/g1_full'),
    ('G2',                 f'{CKPT_ROOT}/g2_full'),
    ('G3',                 f'{CKPT_ROOT}/g3_full'),
    ('Transductive Codex', f'{CKPT_ROOT}/codex_full'),
    ('Channel-Indep',      f'{CKPT_ROOT}/chind_full'),
]

print(f"{'Variant':<25} {'Latest checkpoint'}")
print('-' * 60)
for label, ckpt_dir in VARIANTS:
    ckpts = sorted(glob.glob(f'{ckpt_dir}/epoch_*.pt'))
    if ckpts:
        latest = os.path.basename(ckpts[-1])
        n = len(ckpts)
        print(f'  {label:<23} {latest}  ({n} saved)')
    else:
        print(f'  {label:<23} (no checkpoints)')

## 10. E1 — In-Distribution Linear Probe

PhysioNet MI LOSO: G1 vs. Transductive Codex vs. Channel-Independent.
Fairness anchor for all robustness claims.

In [ ]:
import os, subprocess, sys

os.environ['MNE_DATA'] = MNE_DATA_DIR

# Symlink Drive runs/ into repo so eval scripts find checkpoints via their
# default REPO_ROOT/runs/pretrain/{name}_full/ path.
RUNS_LINK = f'{REPO_DIR}/runs'
RUNS_TARGET = f'{CKPT_ROOT}'
if not os.path.exists(RUNS_LINK):
    os.symlink(RUNS_TARGET, RUNS_LINK)
    print(f'Symlinked {RUNS_LINK} -> {RUNS_TARGET}')
else:
    print(f'runs/ link already exists: {os.path.realpath(RUNS_LINK)}')

In [ ]:
import os
os.environ['MNE_DATA'] = MNE_DATA_DIR

!python {REPO_DIR}/scripts/run_e1.py \
    --device cuda \
    --out-tag full \
    2>&1 | tee {CKPT_ROOT}/e1_log.txt

## 11. E2(a) — Sleep-EDFx cross-session (night 1 → night 2)

Probes G1 and Transductive Codex on Sleep-EDFx within-subject night split.

In [ ]:
import os
os.environ['MNE_DATA'] = MNE_DATA_DIR

!python {REPO_DIR}/scripts/run_e2.py a \
    --device cuda \
    --out-tag full \
    2>&1 | tee {CKPT_ROOT}/e2a_log.txt

## 11b. E2(b) — PhysioNet MI LOSO + Wilcoxon test

LOSO BAC for G1 vs. Codex with paired Wilcoxon signed-rank test.

In [ ]:
import os
os.environ['MNE_DATA'] = MNE_DATA_DIR

!python {REPO_DIR}/scripts/run_e2.py b \
    --device cuda \
    --out-tag full \
    2>&1 | tee {CKPT_ROOT}/e2b_log.txt

## 11c. E2(c) — Cross-montage zero-shot to BCIC-2B

64-ch PhysioNet MI pretrain → 3-ch BCIC-2B evaluation.
Transductive Codex is tested with both `random` and `nn` codex fallbacks.

In [ ]:
import os
os.environ['MNE_DATA'] = MNE_DATA_DIR

!python {REPO_DIR}/scripts/run_e2.py c \
    --device cuda \
    --out-tag full \
    2>&1 | tee {CKPT_ROOT}/e2c_log.txt

## 12. E5 — G1 / G2 / G3 Ablation

Where in attention does g_ij enter? Reports BAC on PhysioNet MI (in-distribution)
and BCIC-2B (cross-montage), plus distinguishing parameter counts.

In [ ]:
import os
os.environ['MNE_DATA'] = MNE_DATA_DIR

!python {REPO_DIR}/scripts/run_e5.py \
    --device cuda \
    --out-tag full \
    2>&1 | tee {CKPT_ROOT}/e5_log.txt

## 13. E7 — Channel-Independent Sanity Check

Does explicit spatial structure help at all? Channel-Independent vs. G1 vs. Codex
on PhysioNet MI (in-distribution) and BCIC-2B (cross-montage).

In [ ]:
import os
os.environ['MNE_DATA'] = MNE_DATA_DIR

!python {REPO_DIR}/scripts/run_e7.py \
    --device cuda \
    --out-tag full \
    2>&1 | tee {CKPT_ROOT}/e7_log.txt

## 14. Full results summary

In [ ]:
import json, glob, os

# ── Checkpoint inventory ──────────────────────────────────────────────────
VARIANTS = [
    ('G1',                 f'{CKPT_ROOT}/g1_full'),
    ('G2',                 f'{CKPT_ROOT}/g2_full'),
    ('G3',                 f'{CKPT_ROOT}/g3_full'),
    ('Transductive Codex', f'{CKPT_ROOT}/codex_full'),
    ('Channel-Indep',      f'{CKPT_ROOT}/chind_full'),
]

print('=== Checkpoint inventory ===')
for label, ckpt_dir in VARIANTS:
    ckpts = sorted(glob.glob(f'{ckpt_dir}/epoch_*.pt'))
    status = os.path.basename(ckpts[-1]) if ckpts else '(missing)'
    print(f'  {label:<25} {status}')

# ── Eval log summary ─────────────────────────────────────────────────────
print()
print('=== Eval logs (last 5 lines each) ===')
for tag, logfile in [
    ('E1',  f'{CKPT_ROOT}/e1_log.txt'),
    ('E2a', f'{CKPT_ROOT}/e2a_log.txt'),
    ('E2b', f'{CKPT_ROOT}/e2b_log.txt'),
    ('E2c', f'{CKPT_ROOT}/e2c_log.txt'),
    ('E5',  f'{CKPT_ROOT}/e5_log.txt'),
    ('E7',  f'{CKPT_ROOT}/e7_log.txt'),
]:
    if os.path.exists(logfile):
        lines = open(logfile).readlines()
        print(f'\n--- {tag} ---')
        print(''.join(lines[-5:]).strip())
    else:
        print(f'\n--- {tag} --- (not run yet)')